In [1]:
import torch 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader,Subset
import torch.nn as nn 

### GoogLeNet Architecture

In [2]:
class Inception(nn.Module):

    def __init__(
        self,
        in_channels,
        c1,
        c3_reduce,
        c3,
        c5_reduce,
        c5,
        pool_proj
    ):
        super().__init__()
        
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=1),
            nn.ReLU(inplace=True)
        )

        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, c3_reduce, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(c3_reduce, c3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, c5_reduce, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(c5_reduce, c5, kernel_size=5, padding=2),
            nn.ReLU(inplace=True)
        )

        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):

        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)

        return torch.cat([b1, b2, b3, b4], dim=1)

In [3]:
class GoogLeNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),
            Inception(
                64,
                c1=32,
                c3_reduce=32,
                c3=64,
                c5_reduce=16,
                c5=16,
                pool_proj=16
            ),
            nn.MaxPool2d(2, 2),

            Inception(
                128,
                c1=64,
                c3_reduce=48,
                c3=96,
                c5_reduce=16,
                c5=32,
                pool_proj=32
            ),

            nn.MaxPool2d(2, 2),

            Inception(
                224,
                c1=64,
                c3_reduce=64,
                c3=96,
                c5_reduce=24,
                c5=32,
                pool_proj=32
            )
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(224, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform
)

# Take only 1,000 training images
train_dataset = Subset(train_dataset, range(1000))

# Take only 1,000 test images
test_dataset = Subset(test_dataset, range(1000))

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

### Model Training

In [5]:
model = GoogLeNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for X, y in train_loader:
        # Forward
        logits = model(X)
        # Loss
        loss = criterion(logits, y)
        # Clear old gradients
        optimizer.zero_grad()
        # Backpropagation
        loss.backward()
        # Update parameters
        optimizer.step()
        
        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {running_loss / len(train_loader):.4f}"
    )

Epoch [1/10], Loss: 2.3002
Epoch [2/10], Loss: 2.2688
Epoch [3/10], Loss: 2.1404
Epoch [4/10], Loss: 2.0705
Epoch [5/10], Loss: 2.0469
Epoch [6/10], Loss: 2.0059
Epoch [7/10], Loss: 1.9597
Epoch [8/10], Loss: 1.8973
Epoch [9/10], Loss: 1.8600
Epoch [10/10], Loss: 1.8356


### Model Evaluation

In [6]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X, y in test_loader:

        logits = model(X)

        predictions = torch.argmax(logits, dim=1)

        total += y.size(0)

        correct += (predictions == y).sum().item()

accuracy = correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 0.307
